# 23-тарау. Қазақша мәтін генераторы

*«Машина қалай үйренеді?» кітабына арналған блокнот. Әр ұяшықтың түсіндірмесі кітапта берілген.*

Ұяшықтарды жоғарыдан төмен қарай ретімен іске қосыңыз: **Shift + Enter**.

## 23.1. Жобаның мақсаты

## 23.2. 1-қадам. Мәтін

**Мәтін керек.** Модельге кез келген қазақша мәтін беріңіз (кемінде 50 000 таңба): ертегілер, Абайдың қара сөздері, өз шығармаларыңыз. Мәтінді `text.txt` атымен сақтап, төмендегі ұяшық арқылы Colab-қа жүктеңіз.

In [ ]:
from google.colab import files
files.upload()   # text.txt файлын таңдаңыз

In [ ]:
import numpy as np
from collections import defaultdict, Counter

text = open("text.txt", encoding="utf-8").read().lower()
chars = sorted(set(text))                   # мәтіндегі барлық таңба
print("Мәтін ұзындығы:", len(text), "таңба")
print("Таңбалар саны:", len(chars))
print(text[:120])

## 23.3. 2-қадам. N-грам моделі

In [ ]:
def build_ngram(text, n):
    model = defaultdict(Counter)
    for i in range(len(text) - n):
        context = text[i:i + n]             # n әріп
        model[context][text[i + n]] += 1    # одан кейінгі әріп
    return model

rng = np.random.default_rng(1)

def generate_ngram(model, n, seed, length=150):
    out = seed
    for _ in range(length):
        counts = model.get(out[-n:])
        if not counts:
            break
        letters = list(counts.keys())
        p = np.array(list(counts.values()), dtype=float)
        out += rng.choice(letters, p=p / p.sum())
    return out

for n, seed in [(2, "ма"), (4, "маши"), (7, "нейронд")]:
    model = build_ngram(text, n)
    print(f"n = {n}: {generate_ngram(model, n, seed)}\n")

## 23.4. 3-қадам. Рекурренттік желілер және LSTM

## 23.5. 4-қадам. Деректерді дайындау

In [ ]:
import keras
from keras import layers

keras.utils.set_random_seed(42)
char2id = {c: i for i, c in enumerate(chars)}     # әріп → нөмір
SEQ_LEN, STEP = 40, 3

X = np.array([[char2id[c] for c in text[i:i + SEQ_LEN]]
              for i in range(0, len(text) - SEQ_LEN, STEP)])
y = np.array([char2id[text[i + SEQ_LEN]]
              for i in range(0, len(text) - SEQ_LEN, STEP)])
print("X:", X.shape, " y:", y.shape)

## 23.6. 5-қадам. LSTM моделін оқыту

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(SEQ_LEN,)),
    layers.Embedding(input_dim=len(chars), output_dim=32),
    layers.LSTM(256),
    layers.Dense(len(chars), activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

h = model.fit(X, y, batch_size=128, epochs=20, verbose=0).history
for e in [1, 5, 10, 15, 20]:
    print(f"Эпоха {e:2d}: қате = {h['loss'][e - 1]:.3f}, "
          f"дәлдік = {h['accuracy'][e - 1]:.3f}")
model.save("char_lstm.keras")      # оқытылған модельді сақтаймыз

## 23.7. 6-қадам. Генерация және температура

In [ ]:
rng = np.random.default_rng(7)

def sample(probs, temperature=1.0):
    logits = np.log(probs + 1e-9) / temperature
    p = np.exp(logits - logits.max())
    return rng.choice(len(p), p=p / p.sum())

def generate(seed, length=160, temperature=1.0):
    out = seed
    for _ in range(length):
        x = np.array([[char2id[c] for c in out[-SEQ_LEN:]]])
        probs = model.predict(x, verbose=0)[0]
        out += chars[sample(probs, temperature)]
    return out

for T in [0.3, 0.7, 1.2]:
    print(f"T = {T}: {generate('жасанды интеллект ', temperature=T)}\n")

In [ ]:
seed = "жасанды интеллек"
x = np.array([[char2id[c] for c in seed.rjust(SEQ_LEN)[-SEQ_LEN:]]])
probs = model.predict(x, verbose=0)[0]
for i in np.argsort(probs)[::-1][:3]:
    print(f"{chars[i]!r}: {probs[i]:.3f}")